# NB15 — Structural Break Test (Chow-type LR Test)

**Purpose:** Address the critique that the dissertation's central non-stationarity claim
rests entirely on visual (shape function) and narrative evidence, with no formal
structural break test applied. This notebook applies a likelihood-ratio Chow test — the
standard generalisation of the Chow test to logistic regression — to the two panel
logistic specifications from NB07B: the term spread regression (the headline
lag1/lag3 sign-reversal finding) and the credit growth regression, split at the 2007
GFC onset.

**Methodology:** The classical Chow test (F-test on RSS) applies to linear models. For a
binary/logistic dependent variable, the standard generalisation is a likelihood-ratio test:
fit the model pooled, then separately pre- and post-break, and test whether the pre/post
coefficients are jointly equal via LR = 2*[(LL_pre + LL_post) - LL_pooled] ~ chi2(k),
where k is the number of parameters (including the constant).

**This notebook does not modify NB07B.** It reuses NB07B's exact data source and
regression specifications, adding only the pre/post subsample fits and the LR test.

**Input:** `df_macro_augmented.csv` (from NB05A) — read-only.


## Cell 1 — Imports and paths

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from pathlib import Path
from statsmodels.tools import add_constant
from statsmodels.discrete.discrete_model import Logit
from scipy import stats

BASE    = Path(r'.')
AUG_DIR = BASE / 'data' / 'processed' / 'augmented_analysis'

df_macro = pd.read_csv(AUG_DIR / 'df_macro_augmented.csv').sort_values(['iso','year']).reset_index(drop=True)
TARGET   = next(c for c in df_macro.columns if 'target' in c.lower())

print(f'Stage 1 macro panel : {df_macro.shape}  ({df_macro["year"].min():.0f}–{df_macro["year"].max():.0f})')
print(f'Target              : {TARGET}')


## Cell 2 — Chow-type likelihood ratio test function (reusable)

In [ ]:
def chow_lr_test(df, feature_cols, target_col, break_year, label=''):
    """Likelihood-ratio Chow test for logistic regression.

    Fits pooled, pre-break, and post-break models on identical feature sets,
    then tests joint equality of coefficients via LR = 2*[(LL_pre+LL_post) - LL_pooled],
    distributed chi2(k) under H0: no structural break.
    """
    reg_df = df[feature_cols + [target_col, 'year']].dropna()
    pre  = reg_df[reg_df['year']  < break_year]
    post = reg_df[reg_df['year'] >= break_year]

    print(f'--- {label} | Break year: {break_year} ---')
    print(f'  Pre-break  ({reg_df["year"].min():.0f}-{break_year-1}): n={len(pre)}, positives={int(pre[target_col].sum())}')
    print(f'  Post-break ({break_year}-{reg_df["year"].max():.0f}): n={len(post)}, positives={int(post[target_col].sum())}')

    if pre[target_col].sum() < 3 or post[target_col].sum() < 3:
        print('  \u26a0\ufe0f  Too few positives in one subsample for reliable Logit \u2014 result may be unstable.')

    try:
        X_pool = add_constant(reg_df[feature_cols]); y_pool = reg_df[target_col]
        X_pre  = add_constant(pre[feature_cols]);    y_pre  = pre[target_col]
        X_post = add_constant(post[feature_cols]);   y_post = post[target_col]

        m_pool = Logit(y_pool, X_pool).fit(disp=0)
        m_pre  = Logit(y_pre,  X_pre ).fit(disp=0)
        m_post = Logit(y_post, X_post).fit(disp=0)

        LL_pool, LL_pre, LL_post = m_pool.llf, m_pre.llf, m_post.llf
        k = len(feature_cols) + 1  # +1 for constant
        LR = 2 * (LL_pre + LL_post - LL_pool)
        p_value = stats.chi2.sf(LR, df=k)

        print(f'  LL(pooled)={LL_pool:.3f}  LL(pre)={LL_pre:.3f}  LL(post)={LL_post:.3f}')
        print(f'  LR statistic = {LR:.3f}   df = {k}   p-value = {p_value:.4f}'
              f'  {"*** structural break detected" if p_value < 0.05 else "(no significant break)"}')
        print()
        print('  Coefficient comparison:')
        for col in feature_cols:
            c_pre, c_post = m_pre.params.get(col, np.nan), m_post.params.get(col, np.nan)
            print(f'    {col}: pre={c_pre:+.4f}  post={c_post:+.4f}  '
                  f'{"(SIGN FLIP)" if np.sign(c_pre)!=np.sign(c_post) else ""}')
        print()
        return {'label': label, 'break_year': break_year, 'LR': LR, 'df': k, 'p_value': p_value}
    except Exception as e:
        print(f'  Model fitting failed: {e}')
        print()
        return {'label': label, 'break_year': break_year, 'LR': np.nan, 'df': np.nan, 'p_value': np.nan}


## Cell 3 — Term spread regression: Chow test at 2007 (primary) and 2008 (sensitivity)

In [ ]:
ts_cols = [c for c in df_macro.columns if 'term_spread' in c and '_dm' in c]
print(f'Term spread columns: {ts_cols}')
print()

results = []
results.append(chow_lr_test(df_macro, ts_cols, TARGET, break_year=2007,
                             label='Term spread (primary break: 2007 GFC onset)'))
results.append(chow_lr_test(df_macro, ts_cols, TARGET, break_year=2008,
                             label='Term spread (sensitivity break: 2008)'))


## Cell 4 — Credit growth regression: confirmatory second test at 2007

In [ ]:
macro_control_cols = [c for c in df_macro.columns if '_lag1_dm' in c and 'tloans' in c][:3]
print(f'Credit growth columns: {macro_control_cols}')
print()

results.append(chow_lr_test(df_macro, macro_control_cols, TARGET, break_year=2007,
                             label='Credit growth (confirmatory, break: 2007)'))


## Cell 5 — Summary table and save

In [ ]:
results_df = pd.DataFrame(results)
print('=' * 90)
print(' STRUCTURAL BREAK TEST — SUMMARY')
print('=' * 90)
print(results_df.to_string(index=False))

results_df.to_csv(AUG_DIR / 'nb15_structural_break_test.csv', index=False)
print(f'\nSaved -> {AUG_DIR / "nb15_structural_break_test.csv"}')
print()
print('Next: paste this table back to Claude to draft the Ch4/Ch5 text.')


## Cell 6 — Completion summary

In [ ]:
print('=' * 65)
print(' NB15 -- STRUCTURAL BREAK TEST COMPLETE')
print('=' * 65)
print()
print('No existing files (NB05A/NB07B outputs) were modified.')
print('Next: paste the printed summary table back to Claude for review before')
print('writing the formal structural break result into Ch4/Ch5.')
